# Project V Milestone 3: GMM Stability and Sensitivity Validation

This notebook stress-tests the Project V M2 Gaussian Mixture Model candidate-rich component without changing the M2 outputs. The validation is label independent: every perturbed run is matched back to the M2 reference component by maximum Jaccard overlap with the 32-star reference set.

In [1]:
# PROJECT_V_REPO_ROOT_FIX
import os
from pathlib import Path

working_directory = Path.cwd().resolve()
repo_candidates = [working_directory, working_directory.parent]
repo_root = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "data" / "processed" / "project_v_m2_cluster_assignments.csv").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root or M2 assignments.")
os.chdir(repo_root)
print("Repository root:", Path.cwd())

Repository root: /Users/liors/Documents/research/gaia-lamost-galactic-archaeology


In [2]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, PowerTransformer

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
FEATURES = ["feh", "rv", "tangential_velocity_kms", "bp_rp", "absolute_g_mag"]
BASELINE_N_COMPONENTS = 9
BASELINE_COVARIANCE = "full"
BASELINE_N_INIT = 5
BASELINE_REG_COVAR = 1e-6
REFERENCE_LABEL = 5
SUBSAMPLE_FRACTION = 0.80

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Matplotlib is building the font cache; this may take a moment.
Python: 3.12.7
scikit-learn: 1.9.0


## Load M2 outputs and define reference groups

In [3]:
processed_dir = Path("data/processed")
figure_dir = Path("figures")
report_dir = Path("report")
for directory in [processed_dir, figure_dir, report_dir]:
    directory.mkdir(parents=True, exist_ok=True)

m2_path = processed_dir / "project_v_m2_cluster_assignments.csv"
candidate_path = processed_dir / "project2_candidate_cross_method_summary.csv"
parent_path = processed_dir / "gaia_lamost_larger_chemo_kinematic_features.csv"

m2 = pd.read_csv(m2_path)
known_candidates = pd.read_csv(candidate_path)
parent = pd.read_csv(parent_path)

assert len(m2) == 1838
assert m2["source_id"].nunique() == len(m2)
assert m2[FEATURES].notna().all().all()
assert set(FEATURES).issubset(parent.columns)

def source_id_key(series):
    return pd.to_numeric(series, errors="raise").astype("Int64").astype(str)

m2 = m2.copy()
m2["source_id_key"] = source_id_key(m2["source_id"])
known_candidates = known_candidates.copy()
known_candidates["source_id_key"] = source_id_key(known_candidates["source_id"])

known_candidate_ids = set(known_candidates["source_id_key"])
assert set(m2.loc[m2["known_candidate"], "source_id_key"]) == known_candidate_ids
assert int(m2["known_candidate"].sum()) == 27

reference_mask = m2["gmm_label"].eq(REFERENCE_LABEL).to_numpy()
reference_ids = set(m2.loc[reference_mask, "source_id_key"])
reference_candidate_ids = set(m2.loc[reference_mask & m2["known_candidate"].to_numpy(), "source_id_key"])
new_member_ids = reference_ids - reference_candidate_ids
omitted_candidate_ids = set(m2.loc[m2["known_candidate"] & m2["gmm_label"].eq(7), "source_id_key"])

assert len(reference_ids) == 32
assert len(reference_candidate_ids) == 24
assert len(new_member_ids) == 8
assert len(omitted_candidate_ids) == 3

global_candidate_rate = float(m2["known_candidate"].mean())
X_raw = m2[FEATURES].copy()
y_m2 = m2["gmm_label"].to_numpy()

print("M2 rows:", len(m2))
print("Reference component label:", REFERENCE_LABEL)
print("Reference members:", len(reference_ids))
print("Reference candidates:", len(reference_candidate_ids))
print("New non-candidate members:", len(new_member_ids))
print("M2 omitted candidates in component 7:", len(omitted_candidate_ids))
print("Global candidate rate:", global_candidate_rate)

M2 rows: 1838
Reference component label: 5
Reference members: 32
Reference candidates: 24
New non-candidate members: 8
M2 omitted candidates in component 7: 3
Global candidate rate: 0.014689880304679


## Baseline reproduction check

In [4]:
baseline_scaler = RobustScaler()
X_baseline = baseline_scaler.fit_transform(X_raw)
baseline_model = GaussianMixture(
    n_components=BASELINE_N_COMPONENTS,
    covariance_type=BASELINE_COVARIANCE,
    n_init=BASELINE_N_INIT,
    random_state=RANDOM_STATE,
    reg_covar=BASELINE_REG_COVAR,
)
baseline_model.fit(X_baseline)
baseline_labels = baseline_model.predict(X_baseline)
baseline_probabilities = baseline_model.predict_proba(X_baseline).max(axis=1)

baseline_ari = adjusted_rand_score(y_m2, baseline_labels)
baseline_reference_ids = set(m2.loc[baseline_labels == REFERENCE_LABEL, "source_id_key"])
baseline_exact_reference_recovery = baseline_reference_ids == reference_ids

print("Adjusted Rand Index versus M2 GMM labels:", baseline_ari)
print("Reference component recovered exactly:", baseline_exact_reference_recovery)
print("Recovered reference size:", len(baseline_reference_ids))
assert np.isclose(baseline_ari, 1.0)
assert baseline_exact_reference_recovery

/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 276, in _count_physical_cores
    cpu_count_physical = int(cpu_info)
                         ^^^^^^^^^^^^^
Adjusted Rand Index versus M2 GMM labels: 1.0
Reference component recovered exactly: True
Recovered reference size: 32


## Validation helpers

In [5]:
def transform_features(frame, feature_list, scaler_name):
    values = frame[feature_list].to_numpy()
    if scaler_name == "RobustScaler":
        scaler = RobustScaler()
        return scaler.fit_transform(values)
    if scaler_name == "StandardScaler":
        scaler = StandardScaler()
        return scaler.fit_transform(values)
    if scaler_name == "MinMaxScaler":
        scaler = MinMaxScaler()
        return scaler.fit_transform(values)
    if scaler_name == "PowerTransformer":
        scaler = PowerTransformer(method="yeo-johnson", standardize=True)
        return scaler.fit_transform(values)
    if scaler_name == "none":
        return values.copy()
    raise ValueError(f"Unknown scaler: {scaler_name}")


def match_component(labels):
    labels = np.asarray(labels)
    best = None
    reference_index = set(np.where(reference_mask)[0])
    for label in sorted(pd.unique(labels)):
        member_index = set(np.where(labels == label)[0])
        intersection = len(reference_index & member_index)
        union = len(reference_index | member_index)
        jaccard = intersection / union if union else 0.0
        recall = intersection / len(reference_index)
        if best is None or (jaccard, recall, -len(member_index)) > (
            best["jaccard_overlap"], best["reference_recall"], -best["matched_group_size"]
        ):
            best = {
                "matched_label": int(label),
                "matched_group_size": int(len(member_index)),
                "reference_overlap_count": int(intersection),
                "jaccard_overlap": float(jaccard),
                "reference_recall": float(recall),
                "selected_indices": member_index,
            }
    return best


def run_gmm_experiment(
    experiment_family,
    experiment_name,
    run_id,
    random_state=RANDOM_STATE,
    train_fraction=1.0,
    features=FEATURES,
    scaler_name="RobustScaler",
    n_components=BASELINE_N_COMPONENTS,
    covariance_type=BASELINE_COVARIANCE,
    n_init=BASELINE_N_INIT,
):
    X = transform_features(X_raw, features, scaler_name)
    n_rows = len(X)
    rng = np.random.default_rng(random_state)
    if train_fraction < 1.0:
        train_size = int(np.floor(train_fraction * n_rows))
        train_index = np.sort(rng.choice(n_rows, size=train_size, replace=False))
    else:
        train_index = np.arange(n_rows)
        train_size = n_rows

    model = GaussianMixture(
        n_components=n_components,
        covariance_type=covariance_type,
        n_init=n_init,
        random_state=random_state,
        reg_covar=BASELINE_REG_COVAR,
    )
    model.fit(X[train_index])
    labels = model.predict(X)
    probabilities = model.predict_proba(X).max(axis=1)
    match = match_component(labels)
    selected = np.array([i in match["selected_indices"] for i in range(n_rows)], dtype=bool)

    candidate_count = int(m2.loc[selected, "known_candidate"].sum())
    candidate_fraction = candidate_count / int(selected.sum()) if int(selected.sum()) else np.nan
    candidate_enrichment = candidate_fraction / global_candidate_rate if global_candidate_rate else np.nan
    ari_vs_m2 = adjusted_rand_score(y_m2, labels)

    run_key = f"{experiment_family}:{experiment_name}:{run_id}"
    summary = {
        "run_key": run_key,
        "experiment_family": experiment_family,
        "experiment_name": experiment_name,
        "run_id": run_id,
        "random_state": random_state,
        "train_fraction": train_fraction,
        "train_size": int(train_size),
        "scaler": scaler_name,
        "features": ";".join(features),
        "n_features": len(features),
        "n_components": n_components,
        "covariance_type": covariance_type,
        "n_init": n_init,
        "converged": bool(model.converged_),
        "n_iter": int(model.n_iter_),
        "lower_bound": float(model.lower_bound_),
        "bic_all": float(model.bic(X)),
        "aic_all": float(model.aic(X)),
        "ari_vs_m2_gmm": float(ari_vs_m2),
        "matched_label": match["matched_label"],
        "matched_group_size": match["matched_group_size"],
        "reference_overlap_count": match["reference_overlap_count"],
        "reference_recall": match["reference_recall"],
        "jaccard_overlap": match["jaccard_overlap"],
        "candidate_count": candidate_count,
        "candidate_fraction": float(candidate_fraction),
        "candidate_enrichment": float(candidate_enrichment),
        "reference_candidates_selected": int(m2.loc[selected & m2["source_id_key"].isin(reference_candidate_ids), "known_candidate"].sum()),
        "new_members_selected": int((selected & m2["source_id_key"].isin(new_member_ids).to_numpy()).sum()),
        "omitted_candidates_selected": int((selected & m2["source_id_key"].isin(omitted_candidate_ids).to_numpy()).sum()),
    }

    star_rows = pd.DataFrame({
        "run_key": run_key,
        "experiment_family": experiment_family,
        "experiment_name": experiment_name,
        "run_id": run_id,
        "source_id": m2["source_id"],
        "source_id_key": m2["source_id_key"],
        "known_candidate": m2["known_candidate"],
        "m2_gmm_label": m2["gmm_label"],
        "run_gmm_label": labels,
        "run_membership_probability": probabilities,
        "selected_in_matched_component": selected,
        "reference_member": m2["source_id_key"].isin(reference_ids).to_numpy(),
        "reference_candidate": m2["source_id_key"].isin(reference_candidate_ids).to_numpy(),
        "new_member": m2["source_id_key"].isin(new_member_ids).to_numpy(),
        "m2_omitted_candidate": m2["source_id_key"].isin(omitted_candidate_ids).to_numpy(),
    })
    return summary, star_rows


def summarize_experiment_family(run_summary):
    metrics = [
        "ari_vs_m2_gmm", "reference_recall", "jaccard_overlap",
        "candidate_fraction", "candidate_enrichment", "matched_group_size",
        "reference_overlap_count", "candidate_count",
        "reference_candidates_selected", "new_members_selected", "omitted_candidates_selected",
    ]
    return (
        run_summary
        .groupby("experiment_family")[metrics]
        .agg(["count", "mean", "std", "min", "median", "max"])
        .reset_index()
    )

## Execute stability and sensitivity suite

In [6]:
experiments = []
experiments.append(dict(experiment_family="baseline_reproduction", experiment_name="m2_locked", run_id="baseline", random_state=42))

for seed in range(30):
    experiments.append(dict(experiment_family="random_seed_stability", experiment_name="seed", run_id=f"seed_{seed:02d}", random_state=seed))

for seed in range(30):
    experiments.append(dict(experiment_family="subsample_80pct", experiment_name="subsample", run_id=f"subsample_{seed:02d}", random_state=1000 + seed, train_fraction=SUBSAMPLE_FRACTION))

for feature in FEATURES:
    reduced_features = [item for item in FEATURES if item != feature]
    experiments.append(dict(experiment_family="feature_ablation", experiment_name=f"drop_{feature}", run_id=f"drop_{feature}", features=reduced_features))

for scaler_name in ["RobustScaler", "StandardScaler", "MinMaxScaler", "PowerTransformer", "none"]:
    experiments.append(dict(experiment_family="scaler_sensitivity", experiment_name=scaler_name, run_id=scaler_name, scaler_name=scaler_name))

for n_components in range(6, 13):
    experiments.append(dict(experiment_family="n_components_sensitivity", experiment_name=f"n_components_{n_components}", run_id=f"n_components_{n_components}", n_components=n_components))

for covariance_type in ["full", "tied", "diag", "spherical"]:
    experiments.append(dict(experiment_family="covariance_sensitivity", experiment_name=covariance_type, run_id=covariance_type, covariance_type=covariance_type))

summary_rows = []
star_tables = []
for i, spec in enumerate(experiments, start=1):
    full_spec = dict(
        experiment_family=spec.get("experiment_family"),
        experiment_name=spec.get("experiment_name"),
        run_id=spec.get("run_id"),
        random_state=spec.get("random_state", RANDOM_STATE),
        train_fraction=spec.get("train_fraction", 1.0),
        features=spec.get("features", FEATURES),
        scaler_name=spec.get("scaler_name", "RobustScaler"),
        n_components=spec.get("n_components", BASELINE_N_COMPONENTS),
        covariance_type=spec.get("covariance_type", BASELINE_COVARIANCE),
        n_init=spec.get("n_init", BASELINE_N_INIT),
    )
    summary, star_rows = run_gmm_experiment(**full_spec)
    summary_rows.append(summary)
    star_tables.append(star_rows)
    if i % 10 == 0 or i == len(experiments):
        print(f"Completed {i}/{len(experiments)} runs")

run_summary = pd.DataFrame(summary_rows)
star_by_experiment = pd.concat(star_tables, ignore_index=True)

print("Runs:", len(run_summary))
run_summary.groupby("experiment_family").size().to_frame("run_count")

Completed 10/82 runs
Completed 20/82 runs
Completed 30/82 runs
Completed 40/82 runs
Completed 50/82 runs
Completed 60/82 runs
Completed 70/82 runs
Completed 80/82 runs
Completed 82/82 runs
Runs: 82


## Aggregate star and group stability

In [7]:
star_stability = (
    star_by_experiment
    .groupby(["source_id", "source_id_key", "known_candidate", "m2_gmm_label", "reference_member", "reference_candidate", "new_member", "m2_omitted_candidate"], as_index=False)
    .agg(
        selection_count=("selected_in_matched_component", "sum"),
        selection_frequency=("selected_in_matched_component", "mean"),
        mean_membership_probability=("run_membership_probability", "mean"),
        median_membership_probability=("run_membership_probability", "median"),
    )
)
star_stability["stability_group"] = np.select(
    [
        star_stability["reference_candidate"],
        star_stability["new_member"],
        star_stability["m2_omitted_candidate"],
        star_stability["known_candidate"],
        star_stability["reference_member"],
    ],
    [
        "24 M2 reference candidates",
        "8 M2 new members",
        "3 M2 omitted candidates",
        "other known candidates",
        "other reference members",
    ],
    default="field stars",
)

star_stability_by_experiment = star_by_experiment.copy()
star_stability_by_experiment["stability_group"] = np.select(
    [
        star_stability_by_experiment["reference_candidate"],
        star_stability_by_experiment["new_member"],
        star_stability_by_experiment["m2_omitted_candidate"],
        star_stability_by_experiment["known_candidate"],
        star_stability_by_experiment["reference_member"],
    ],
    [
        "24 M2 reference candidates",
        "8 M2 new members",
        "3 M2 omitted candidates",
        "other known candidates",
        "other reference members",
    ],
    default="field stars",
)

group_stability_summary = (
    star_stability
    .groupby("stability_group", as_index=False)
    .agg(
        star_count=("source_id", "count"),
        mean_selection_frequency=("selection_frequency", "mean"),
        min_selection_frequency=("selection_frequency", "min"),
        median_selection_frequency=("selection_frequency", "median"),
        max_selection_frequency=("selection_frequency", "max"),
    )
    .sort_values("mean_selection_frequency", ascending=False)
)

experiment_summary = (
    run_summary
    .groupby("experiment_family", as_index=False)
    .agg(
        run_count=("run_key", "count"),
        mean_ari_vs_m2_gmm=("ari_vs_m2_gmm", "mean"),
        min_ari_vs_m2_gmm=("ari_vs_m2_gmm", "min"),
        mean_reference_recall=("reference_recall", "mean"),
        min_reference_recall=("reference_recall", "min"),
        mean_jaccard_overlap=("jaccard_overlap", "mean"),
        min_jaccard_overlap=("jaccard_overlap", "min"),
        mean_candidate_fraction=("candidate_fraction", "mean"),
        mean_candidate_enrichment=("candidate_enrichment", "mean"),
        min_candidate_enrichment=("candidate_enrichment", "min"),
        mean_matched_group_size=("matched_group_size", "mean"),
        min_matched_group_size=("matched_group_size", "min"),
        max_matched_group_size=("matched_group_size", "max"),
    )
)

run_summary.sort_values(["experiment_family", "run_id"]).head()

## Persist CSV outputs

In [8]:
run_summary_path = processed_dir / "project_v_m3_run_summary.csv"
star_stability_path = processed_dir / "project_v_m3_star_stability.csv"
star_stability_by_experiment_path = processed_dir / "project_v_m3_star_stability_by_experiment.csv"
group_stability_summary_path = processed_dir / "project_v_m3_group_stability_summary.csv"
experiment_summary_path = processed_dir / "project_v_m3_experiment_summary.csv"

run_summary.to_csv(run_summary_path, index=False)
star_stability.to_csv(star_stability_path, index=False)
star_stability_by_experiment.to_csv(star_stability_by_experiment_path, index=False)
group_stability_summary.to_csv(group_stability_summary_path, index=False)
experiment_summary.to_csv(experiment_summary_path, index=False)

for path in [run_summary_path, star_stability_path, star_stability_by_experiment_path, group_stability_summary_path, experiment_summary_path]:
    print("Saved:", path, pd.read_csv(path).shape)

Saved: data/processed/project_v_m3_run_summary.csv (82, 30)
Saved: data/processed/project_v_m3_star_stability.csv (1838, 13)
Saved: data/processed/project_v_m3_star_stability_by_experiment.csv (150716, 16)
Saved: data/processed/project_v_m3_group_stability_summary.csv (4, 6)
Saved: data/processed/project_v_m3_experiment_summary.csv (7, 14)


## Figures

In [9]:
summary_order = [
    "baseline_reproduction", "random_seed_stability", "subsample_80pct", "feature_ablation",
    "scaler_sensitivity", "n_components_sensitivity", "covariance_sensitivity",
]
plot_summary = run_summary.copy()
plot_summary["experiment_family"] = pd.Categorical(plot_summary["experiment_family"], categories=summary_order, ordered=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=plot_summary, x="experiment_family", y="reference_recall", ax=axes[0, 0], color="#8ecae6")
sns.stripplot(data=plot_summary, x="experiment_family", y="reference_recall", ax=axes[0, 0], color="#023047", size=4, alpha=0.65)
axes[0, 0].set_title("Reference recall by validation family")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Reference recall")
axes[0, 0].tick_params(axis="x", rotation=35)

sns.boxplot(data=plot_summary, x="experiment_family", y="jaccard_overlap", ax=axes[0, 1], color="#b7e4c7")
sns.stripplot(data=plot_summary, x="experiment_family", y="jaccard_overlap", ax=axes[0, 1], color="#1b4332", size=4, alpha=0.65)
axes[0, 1].set_title("Jaccard overlap by validation family")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("Jaccard overlap")
axes[0, 1].tick_params(axis="x", rotation=35)

sns.boxplot(data=plot_summary, x="experiment_family", y="candidate_enrichment", ax=axes[1, 0], color="#ffb703")
sns.stripplot(data=plot_summary, x="experiment_family", y="candidate_enrichment", ax=axes[1, 0], color="#7f4f24", size=4, alpha=0.65)
axes[1, 0].axhline(51.06, color="#d62828", linestyle="--", linewidth=1.2, label="M2 51.06x")
axes[1, 0].set_title("Candidate enrichment by validation family")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("Candidate enrichment")
axes[1, 0].tick_params(axis="x", rotation=35)
axes[1, 0].legend()

sns.scatterplot(data=plot_summary, x="matched_group_size", y="candidate_fraction", hue="experiment_family", ax=axes[1, 1], s=65)
axes[1, 1].set_title("Matched component size and candidate concentration")
axes[1, 1].set_xlabel("Matched component size")
axes[1, 1].set_ylabel("Candidate fraction")
axes[1, 1].legend(fontsize=8, loc="best")

fig.suptitle("Project V M3 GMM stability and sensitivity validation", fontsize=16, y=1.01)
fig.tight_layout()
validation_summary_path = figure_dir / "project_v_m3_validation_summary.png"
fig.savefig(validation_summary_path, dpi=180, bbox_inches="tight")
plt.show()

focused = star_stability[star_stability["stability_group"].isin([
    "24 M2 reference candidates", "8 M2 new members", "3 M2 omitted candidates"
])].copy()
focused["source_id_short"] = focused["source_id_key"].str[-6:]
focused = focused.sort_values(["stability_group", "selection_frequency", "source_id_key"], ascending=[True, False, True])

fig2, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=focused, x="source_id_short", y="selection_frequency", hue="stability_group", dodge=False, ax=ax, palette=["#2a9d8f", "#e76f51", "#577590"])
ax.set_ylim(0, 1.05)
ax.set_title("Per-star recovery frequency for reference candidates, new members, and omitted candidates")
ax.set_xlabel("Source ID suffix")
ax.set_ylabel("Selection frequency across all M3 runs")
ax.tick_params(axis="x", rotation=70)
ax.legend(loc="lower left", fontsize=9)
fig2.tight_layout()
per_star_path = figure_dir / "project_v_m3_per_star_recovery.png"
fig2.savefig(per_star_path, dpi=180, bbox_inches="tight")
plt.show()

spec_families = ["feature_ablation", "scaler_sensitivity", "n_components_sensitivity", "covariance_sensitivity"]
spec_plot = run_summary[run_summary["experiment_family"].isin(spec_families)].copy()
spec_plot["label"] = spec_plot["experiment_name"]

fig3, axes = plt.subplots(1, 3, figsize=(17, 5.5))
sns.barplot(data=spec_plot, x="label", y="jaccard_overlap", hue="experiment_family", ax=axes[0])
axes[0].set_title("Jaccard under model specification changes")
axes[0].set_xlabel("")
axes[0].set_ylabel("Jaccard overlap")
axes[0].tick_params(axis="x", rotation=70)
axes[0].legend(fontsize=8)

sns.barplot(data=spec_plot, x="label", y="reference_recall", hue="experiment_family", ax=axes[1])
axes[1].set_title("Reference recall under model specification changes")
axes[1].set_xlabel("")
axes[1].set_ylabel("Reference recall")
axes[1].tick_params(axis="x", rotation=70)
axes[1].legend_.remove()

sns.barplot(data=spec_plot, x="label", y="candidate_enrichment", hue="experiment_family", ax=axes[2])
axes[2].axhline(51.06, color="#d62828", linestyle="--", linewidth=1.2)
axes[2].set_title("Candidate enrichment under model specification changes")
axes[2].set_xlabel("")
axes[2].set_ylabel("Candidate enrichment")
axes[2].tick_params(axis="x", rotation=70)
axes[2].legend_.remove()

fig3.suptitle("Project V M3 model specification sensitivity", fontsize=16, y=1.03)
fig3.tight_layout()
model_spec_path = figure_dir / "project_v_m3_model_specification_sensitivity.png"
fig3.savefig(model_spec_path, dpi=180, bbox_inches="tight")
plt.show()

for path in [validation_summary_path, per_star_path, model_spec_path]:
    print("Saved:", path, path.stat().st_size)

notebooks/27_project_v_gmm_stability_sensitivity_validation.ipynb:cell16:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/27_project_v_gmm_stability_sensitivity_validation.ipynb:cell16:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/27_project_v_gmm_stability_sensitivity_validation.ipynb:cell16:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Saved: figures/project_v_m3_validation_summary.png 440868
Saved: figures/project_v_m3_per_star_recovery.png 164899
Saved: figures/project_v_m3_model_specification_sensitivity.png 248429


## Key numerical summaries

In [10]:
display(experiment_summary)
display(group_stability_summary)

key_metrics = {
    "baseline_ari": float(baseline_ari),
    "baseline_exact_reference_recovery": bool(baseline_exact_reference_recovery),
    "m2_reference_size": len(reference_ids),
    "m2_reference_candidates": len(reference_candidate_ids),
    "m2_new_members": len(new_member_ids),
    "m2_omitted_candidates_component_7": len(omitted_candidate_ids),
    "all_runs": len(run_summary),
    "random_seed_min_jaccard": float(run_summary.loc[run_summary["experiment_family"].eq("random_seed_stability"), "jaccard_overlap"].min()),
    "random_seed_mean_jaccard": float(run_summary.loc[run_summary["experiment_family"].eq("random_seed_stability"), "jaccard_overlap"].mean()),
    "subsample_min_jaccard": float(run_summary.loc[run_summary["experiment_family"].eq("subsample_80pct"), "jaccard_overlap"].min()),
    "subsample_mean_jaccard": float(run_summary.loc[run_summary["experiment_family"].eq("subsample_80pct"), "jaccard_overlap"].mean()),
    "specification_min_jaccard": float(run_summary.loc[run_summary["experiment_family"].isin(["feature_ablation", "scaler_sensitivity", "n_components_sensitivity", "covariance_sensitivity"]), "jaccard_overlap"].min()),
    "specification_mean_jaccard": float(run_summary.loc[run_summary["experiment_family"].isin(["feature_ablation", "scaler_sensitivity", "n_components_sensitivity", "covariance_sensitivity"]), "jaccard_overlap"].mean()),
}
print(json.dumps(key_metrics, indent=2, sort_keys=True))

          experiment_family  run_count  mean_ari_vs_m2_gmm  min_ari_vs_m2_gmm  mean_reference_recall  min_reference_recall  mean_jaccard_overlap  min_jaccard_overlap  mean_candidate_fraction  mean_candidate_enrichment  min_candidate_enrichment  mean_matched_group_size  min_matched_group_size  max_matched_group_size
0     baseline_reproduction          1            1.000000           1.000000               1.000000               1.00000              1.000000             1.000000                 0.750000                  51.055556                 51.055556                32.000000                      32                      32
1    covariance_sensitivity          4            0.464821           0.246250               0.750000               0.43750              0.701042             0.437500                 0.839372                  57.139470                 47.273663                26.250000                      14                      36
2          feature_ablation          5           

## Scientific report

In [11]:
def fmt(value, digits=3):
    if pd.isna(value):
        return "nan"
    return f"{value:.{digits}f}"

baseline_run = run_summary[run_summary["experiment_family"].eq("baseline_reproduction")].iloc[0]
seed_runs = run_summary[run_summary["experiment_family"].eq("random_seed_stability")]
sub_runs = run_summary[run_summary["experiment_family"].eq("subsample_80pct")]
spec_runs = run_summary[run_summary["experiment_family"].isin(["feature_ablation", "scaler_sensitivity", "n_components_sensitivity", "covariance_sensitivity"])]

report_text = f"""# Project V Milestone 3: GMM Stability and Sensitivity Validation

## Purpose

This milestone tests whether the Project V M2 GMM component enriched in external candidates is numerically stable and how strongly it depends on modeling choices. The M2 outputs are not modified. Every perturbed model is matched to the M2 reference component with a label-independent maximum Jaccard overlap against the 32-star M2 reference set.

## M2 Baseline Reproduction

The code-level M2 baseline is `RobustScaler` on `feh`, `rv`, `tangential_velocity_kms`, `bp_rp`, and `absolute_g_mag`, followed by `GaussianMixture(n_components=9, covariance_type=full, n_init=5, random_state=42, reg_covar=1e-6)`. Re-running that specification reproduced the M2 labels with Adjusted Rand Index {baseline_ari:.6f}. The 32-star reference component was recovered exactly.

The reference component contains 24 of the 27 known Project II candidates and 8 non-candidate stars newly grouped with them by M2. The remaining 3 known candidates fall in M2 GMM component 7.

## Validation Design

The validation suite includes {len(run_summary)} total GMM fits: 30 random-seed runs, 30 independent 80 percent no-replacement subsample fits, five one-feature ablations, five scaling choices, seven component-count choices from 6 to 12, and four covariance structures. For each run, the matched component is selected by maximum Jaccard overlap with the 32-star M2 reference component, avoiding any dependence on arbitrary GMM label numbering.

## Main Results

Random-seed stability is strong: mean Jaccard overlap is {fmt(seed_runs['jaccard_overlap'].mean())}, minimum Jaccard overlap is {fmt(seed_runs['jaccard_overlap'].min())}, mean reference recall is {fmt(seed_runs['reference_recall'].mean())}, and mean candidate enrichment is {fmt(seed_runs['candidate_enrichment'].mean(), 2)}x.

The 80 percent subsample test is less exact but still recovers much of the M2 structure: mean Jaccard overlap is {fmt(sub_runs['jaccard_overlap'].mean())}, minimum Jaccard overlap is {fmt(sub_runs['jaccard_overlap'].min())}, mean reference recall is {fmt(sub_runs['reference_recall'].mean())}, and mean candidate enrichment is {fmt(sub_runs['candidate_enrichment'].mean(), 2)}x.

Across feature, scaler, component-count, and covariance sensitivity runs, the matched structure is model-setting dependent: mean Jaccard overlap is {fmt(spec_runs['jaccard_overlap'].mean())}, minimum Jaccard overlap is {fmt(spec_runs['jaccard_overlap'].min())}, mean reference recall is {fmt(spec_runs['reference_recall'].mean())}, and mean candidate enrichment is {fmt(spec_runs['candidate_enrichment'].mean(), 2)}x.

## Per-Star Stability

The group-level stability table shows that the 24 M2 reference candidates have mean selection frequency {fmt(group_stability_summary.loc[group_stability_summary['stability_group'].eq('24 M2 reference candidates'), 'mean_selection_frequency'].iloc[0])}. The 8 M2 new members have mean selection frequency {fmt(group_stability_summary.loc[group_stability_summary['stability_group'].eq('8 M2 new members'), 'mean_selection_frequency'].iloc[0])}. The 3 M2 omitted candidates have mean selection frequency {fmt(group_stability_summary.loc[group_stability_summary['stability_group'].eq('3 M2 omitted candidates'), 'mean_selection_frequency'].iloc[0])}, which confirms that the matched M2-like component usually remains distinct from the component-7 candidate residuals.

## Interpretation

The M2 51.06x candidate enrichment is best described as **model-setting dependent rather than universally stable**. It is computationally reproducible under the exact M2 configuration and robust to many random initializations, but it weakens or changes under subsampling and under some reasonable specification changes. This means the M2 component is a useful candidate-rich chemo-kinematic signal for follow-up prioritization, not proof of a physically real stellar substructure by itself.

The scientific interpretation should therefore separate two claims. The computational claim is strong for exact reproducibility and random-seed stability. The physical claim remains provisional and requires independent validation with orbital actions, abundances beyond `[Fe/H]`, selection-function checks, and external spectroscopy.

## Outputs

- `data/processed/project_v_m3_run_summary.csv`
- `data/processed/project_v_m3_star_stability.csv`
- `data/processed/project_v_m3_star_stability_by_experiment.csv`
- `data/processed/project_v_m3_group_stability_summary.csv`
- `data/processed/project_v_m3_experiment_summary.csv`
- `figures/project_v_m3_validation_summary.png`
- `figures/project_v_m3_per_star_recovery.png`
- `figures/project_v_m3_model_specification_sensitivity.png`
"""

report_path = report_dir / "project_v_milestone3_gmm_stability_sensitivity_validation.md"
report_path.write_text(report_text)
print("Saved:", report_path)
print(report_text[:1200])

Saved: report/project_v_milestone3_gmm_stability_sensitivity_validation.md
# Project V Milestone 3: GMM Stability and Sensitivity Validation

## Purpose

This milestone tests whether the Project V M2 GMM component enriched in external candidates is numerically stable and how strongly it depends on modeling choices. The M2 outputs are not modified. Every perturbed model is matched to the M2 reference component with a label-independent maximum Jaccard overlap against the 32-star M2 reference set.

## M2 Baseline Reproduction

The code-level M2 baseline is `RobustScaler` on `feh`, `rv`, `tangential_velocity_kms`, `bp_rp`, and `absolute_g_mag`, followed by `GaussianMixture(n_components=9, covariance_type=full, n_init=5, random_state=42, reg_covar=1e-6)`. Re-running that specification reproduced the M2 labels with Adjusted Rand Index 1.000000. The 32-star reference component was recovered exactly.

The reference component contains 24 of the 27 known Project II candidates and 8 non-candidate

## Output integrity checks

In [12]:
csv_paths = [
    run_summary_path,
    star_stability_path,
    star_stability_by_experiment_path,
    group_stability_summary_path,
    experiment_summary_path,
]
figure_paths = [validation_summary_path, per_star_path, model_spec_path]

for path in csv_paths:
    table = pd.read_csv(path)
    assert len(table) > 0, path
    print("CSV OK:", path, table.shape)

from PIL import Image
for path in figure_paths:
    assert path.exists() and path.stat().st_size > 10_000, path
    with Image.open(path) as image:
        width, height = image.size
        extrema = image.convert("L").getextrema()
    assert width > 500 and height > 300, (path, width, height)
    assert extrema[0] != extrema[1], (path, extrema)
    print("Figure OK:", path, f"{width}x{height}", "bytes", path.stat().st_size)

assert baseline_exact_reference_recovery
assert np.isclose(baseline_ari, 1.0)
print("All M3 output checks passed.")

CSV OK: data/processed/project_v_m3_run_summary.csv (82, 30)
CSV OK: data/processed/project_v_m3_star_stability.csv (1838, 13)
CSV OK: data/processed/project_v_m3_star_stability_by_experiment.csv (150716, 16)
CSV OK: data/processed/project_v_m3_group_stability_summary.csv (4, 6)
CSV OK: data/processed/project_v_m3_experiment_summary.csv (7, 14)
Figure OK: figures/project_v_m3_validation_summary.png 2480x1822 bytes 440868
Figure OK: figures/project_v_m3_per_star_recovery.png 2491x1232 bytes 164899
Figure OK: figures/project_v_m3_model_specification_sensitivity.png 3028x1023 bytes 248429
All M3 output checks passed.
